# libgltf Kotlin 代码审查
扫描 `src/main/kotlin`，输出规模、重复、遗留旧前缀、TODO 等指标。
在 IntelliJ 的 Kotlin Notebook 中打开本文件并运行全部单元格。

In [ ]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.Paths

val root: Path = Paths.get(System.getProperty("user.dir")).resolve("src/main/kotlin")
val kotlinFiles: List<Path> = Files.walk(root).use { paths ->
    paths.filter { Files.isRegularFile(it) && it.fileName.toString().endsWith(".kt") }.toList()
}
println("Kotlin 文件数: ${kotlinFiles.size}")

In [ ]:
data class FileInfo(val path: Path, val lines: Int, val content: String)

val infos: List<FileInfo> = kotlinFiles.map { f ->
    val text = Files.readString(f)
    FileInfo(f, text.lines().size, text)
}

val totalLines = infos.sumOf { it.lines }
println("总行数: $totalLines")
println("\n最长的 10 个文件:")
infos.sortedByDescending { it.lines }.take(10).forEach { info ->
    println("  %5d  %s".format(info.lines, info.path))
}

In [ ]:
val topLevelPattern = Regex("^(?:class|object|interface|enum class|data class|sealed class|abstract class)\\s+([A-Za-z0-9_]+)")
val names = infos.flatMap { info ->
    info.content.lines().mapNotNull { topLevelPattern.find(it)?.groupValues?.get(1) }
}

println("顶层声明数: ${names.size}")
val duplicates = names.groupingBy { it }.eachCount().filterValues { it > 1 }
println("重复简单类名: ${if (duplicates.isEmpty()) "无" else duplicates}")
println("\n仍带 Gltf 前缀的类:")
names.filter { it.startsWith("Gltf") }.sorted().forEach { println("  $it") }

In [ ]:
val todo = Regex("(?i)(TODO|FIXME|XXX|HACK)")
val findings = infos.flatMap { info ->
    info.content.lines().mapIndexedNotNull { index, line ->
        if (todo.containsMatchIn(line)) "${info.path}:${index + 1}: ${line.trim()}" else null
    }
}
println("TODO/FIXME 标记: ${findings.size}")
findings.forEach { println("  $it") }

val oldPrefix = Regex("\\bGltf[A-Z][A-Za-z0-9_]*\\b")
val oldRefs = infos.sumOf { info -> oldPrefix.findAll(info.content).count() }
println("遗留 Gltf 前缀引用数: $oldRefs")